# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [5]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [6]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [7]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [8]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
- Why this question matters:
- Tables/files used:
- Row meaning of each important table:
- Early data quality issues:
- Initial cleaning plan:
- Possible target/outcome:
- Ethics, privacy, or governance concerns:



In [ ]:
# Business Problem: 
# A lot goes in to making a movie and there are many different aspects of a movie, from genre, director, actors, special effects, budget, and 
# many more. I want to look into what movie attributes are associated with higher box office revenue and if there are any underlying attributes 
# high-grossing movies might have in common. Movie studios invest millions of dollars into making a movie. If movie studios could better understand
# which movie characteristics are associated with higher box office revenue, it may help them make better decisions about which films to make in
# the future.
# 
# Dataset Description and Row Meaning: 
# We will focus on one column from the movies_metadata file, which is "Revenue". We will use many other
# columms from this table as well as credits_curated, keywords_curated, and ratings. 
# From movies_metadata, one row represents a certain movie by it's movie ID. The columns include information about that movie. We will focus on many 
# columns in this table including Revenue, Release Date, Runtime, Budget, Popularity, Vote Average, Vote Count, and Genre. 
# From credits_curated, one row represents a movie by it's tmdbId. We will focus on the director and the cast list from this table.
# From keywords_curated, one row represents a movie by it's tmdbId and gives keywords describing that movie.
# From ratings, one row represents the rating given to a movie by one user. We will use the average user rating for a movie. We will also look at
# how many ratings that movie had. 
# 
# Initial Data Profiling:
# The shape of the five tables is as follows:
# movies: (9082, 21)
#ratings: (99810, 5)
#links: (9125, 3)
#credits: (9082, 3)
#keywords: (9082, 2)
# We have run analysis to see what important columns in each table may be missing data. Missing data can skew our analysis and we will have to decide 
# if we are to compute the missing data, look up missing data on alternate sites, or remove lines with needed missing data. We have also checked 
# for any suspicious values. For example, we found that 4,322 movies have a revenue of $0 as opposed to just a Null value. Also, 4.443 movies have 
# a budget of $0, and 20 movies have a runtime of 0 minutes. This can also skew the data and we will have to decide how we proceed with these fields
# as well. We have checked for duplicate tmdbId's, imdbId's, and movieId's in movies, credits, and keywords to identify duplicates. No vital duplicates
# were found. 

# Initial Cleaning Plan and Target Outcome:
# We will begin cleaning the data by deciding how to handle missing values. Columns such as revenue and budget have over 4,000 lines each with $0 value,
# which is too many lines to look up manually. It is also too many lines to omit. We can find an average for these lines by comparing genre, director,
# and release date to generate these values. We will clean up the keywords column and give each keyword it's own column for easier analysis. We will also
# aggregate the ratings to the movies. As one movie may have multiple ratings, we will average the ratings for each movie and define a minimum rating
# count to ensure accurate data. We will then merge and join the tables into one clean data table for analysis. Our target objective is to identify which
# which movie attributes are associated with high box office grosses for movies. Connecting relationships between attributes can help explain why some
# movies are more profitable than others.

# Ethics and Privacy Concerns:
# Our project uses publicly available data regarding movies, cast, and ratings. Ratings are specific to a user, but no personally identifiable information
# is present per user, just an assigned userId. Final analysis may be biased due to missing information which may be computed rather than being true
# values. Highly popular movies may receive more ratings and a larger marketing budget than smaller independent films which can also alter results.
# Repository Structure
#movie-box-office-project/
#│
#├── ban6003-movies-term-project-KevinMitchell
#├── data/
#│ ├── credits_curated.csv
#│ ├── keywords_curated.csv
#│ ├── links.csv
#│ ├── movies_metadata.csv
#│ └── ratings.csv
#| └── README.md 
#│
#├── notebooks/
#│ └── movies_project_starter.ipynb
# README.md has been updated with data sources, repository structure, and reproduction instructions.





In [9]:
for name, df in {
    "movies": movies,
    "ratings": ratings,
    "links": links,
    "credits": credits,
    "keywords": keywords
}.items():
    print(f"{name}: {df.shape}")

columns_to_check1 = [
'revenue',
'release_date',
'runtime',
'budget',
'popularity',
'vote_average',
'vote_count',
'genres'
]
print(movies[columns_to_check1].isna().sum())
for col in columns_to_check1:
    print(f"\n{col}")
    print("Missing:", movies[col].isna().sum())

columns_to_check2 = [
'director',
'top_cast'
]
print(credits[columns_to_check2].isna().sum())
for col in columns_to_check2:
    print(f"\n{col}")
    print("Missing:", credits[col].isna().sum())

columns_to_check3 = [
'tmdbId',
'keywords_list'
]
print(keywords[columns_to_check3].isna().sum())
for col in columns_to_check3:
    print(f"\n{col}")
    print("Missing:", keywords[col].isna().sum())

columns_to_check4 = [
'movieId',
'imdbId',
'tmdbId'
]
print(links[columns_to_check4].isna().sum())
for col in columns_to_check4:
    print(f"\n{col}")
    print("Missing:", links[col].isna().sum())

columns_to_check5 = [
'userId',
'movieId',
'rating'
]
print(ratings[columns_to_check5].isna().sum())
for col in columns_to_check5:
    print(f"\n{col}")
    print("Missing:", ratings[col].isna().sum())

for col in ['revenue', 'budget', 'runtime', 'vote_count']:
    print(f"\n{col}")
    print("Zero Values:", (movies[col] == 0).sum())

print("MOVIES")
print("movieId duplicates:", movies['movieId'].duplicated().sum())
print("tmdbId duplicates:", movies['tmdbId'].duplicated().sum())
print("CREDITS")
print("tmdbId duplicates:", credits['tmdbId'].duplicated().sum())
print("KEYWORDS")
print("tmdbId duplicates:", keywords['tmdbId'].duplicated().sum())

movies: (9082, 21)
ratings: (99810, 5)
links: (9125, 3)
credits: (9082, 3)
keywords: (9082, 2)
revenue         0
release_date    0
runtime         0
budget          0
popularity      0
vote_average    0
vote_count      0
genres          0
dtype: int64

revenue
Missing: 0

release_date
Missing: 0

runtime
Missing: 0

budget
Missing: 0

popularity
Missing: 0

vote_average
Missing: 0

vote_count
Missing: 0

genres
Missing: 0
director    24
top_cast    90
dtype: int64

director
Missing: 24

top_cast
Missing: 90
tmdbId             0
keywords_list    761
dtype: int64

tmdbId
Missing: 0

keywords_list
Missing: 761
movieId     0
imdbId      0
tmdbId     13
dtype: int64

movieId
Missing: 0

imdbId
Missing: 0

tmdbId
Missing: 13
userId     0
movieId    0
rating     0
dtype: int64

userId
Missing: 0

movieId
Missing: 0

rating
Missing: 0

revenue
Zero Values: 4322

budget
Zero Values: 4443

runtime
Zero Values: 20

vote_count
Zero Values: 44
MOVIES
movieId duplicates: 0
tmdbId duplicates: 0
CREDI

In [10]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


,missing_rate
userId,0.0
movieId,0.0
rating,0.0
timestamp,0.0
rating_datetime,0.0


... (5 outputs left out)

,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [25]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.

#One row represents one movie. MovieID is the Primary key used to identify each unique movie in the ABT.
#Aggregation was performed with avg_rating and rating_caount to create ratings_agg as the ratings data was previously given with
#multiple lines per movie. 
#We joined credits_curated, keywords_curated, and ratings_agg to our base table of movies_metadata

ratings_summary = (
    ratings
    .groupby("movieId")
    .agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count"),
        rating_std=("rating", "std")
    )
    .reset_index()
)

abt = movies.copy()

print("Starting rows:", len(abt))
print("Duplicate movieIds:", abt["movieId"].duplicated().sum())

abt = abt.merge(
        credits,
        on="tmdbId",
        how="left"
    )
print("\nAfter credits merge")
print("Rows:", len(abt))
print("Duplicate movieIds:", abt["movieId"].duplicated().sum())

abt = abt.merge(
        keywords,
        on="tmdbId",
        how="left"
    )
print("\nAfter keywords merge")
print("Rows:", len(abt))
print("Duplicate movieIds:", abt["movieId"].duplicated().sum())

abt = abt.merge(
        ratings_summary,
        on="movieId",
        how="left"
    )

print("\nAfter ratings summary merge")
print("Rows:", len(abt))
print("Duplicate movieIds:", abt["movieId"].duplicated().sum())

Starting rows: 9082
Duplicate movieIds: 0

After credits merge
Rows: 9082
Duplicate movieIds: 0

After keywords merge
Rows: 9082
Duplicate movieIds: 0

After ratings summary merge
Rows: 9082
Duplicate movieIds: 0


In [27]:
print("Movies rows:", len(movies))
print("ABT rows:", len(abt))

print("\nDuplicate movieIds:",
      abt["movieId"].duplicated().sum())

Movies rows: 9082
ABT rows: 9082

Duplicate movieIds: 0


We created our ABT and performed validation checks after we merged the tables to confirm that we did not create or remove any rows. We performed duplicate checks on the movieID and tmdbID and also confirmed the number of movies that contained rating information to confirm that our ABT integration was successful. We then performed a missingness assessment to indentify columns with missing or incomplete information. 



In [28]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_ratings": abt["rating_count"].notna().sum()
})

display(abt_validation)
display(abt.head())

display(
    abt.isna()
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("missing_rate")
)

movie_rows_before_merges    9082
abt_rows_after_merges       9082
duplicate_movie_ids            0
duplicate_tmdb_ids             0
movies_with_ratings         9025
dtype: int64

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,director,top_cast,keywords_list,avg_rating,rating_count,rating_std
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,3.872470,247.0,0.958981
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,3.401869,107.0,0.880714
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,3.161017,59.0,1.150115
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,2.384615,13.0,0.938835
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,3.267857,56.0,0.948512


,missing_rate
rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
avg_rating,0.006276
rating_count,0.006276
genres_list,0.003854
primary_genre,0.003854
director,0.002643
status,0.000220
tmdbId,0.000000


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


We performed an SQL validation query on our ABT. This confirmed the row count and verified that no rows were added after we performed our joins. It also show us any missing values for revenue, budget, average ratings, ratings counts, director, and keywords. 

In [34]:
import sqlite3

with sqlite3.connect(":memory:") as connection:

    abt.to_sql(
        "movies_abt",
        connection,
        if_exists="replace",
        index=False
    )

    sql_validation = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT movieId) AS unique_movie_ids,
            COUNT(DISTINCT tmdbId) AS unique_tmdb_ids,

            SUM(CASE WHEN revenue IS NULL THEN 1 ELSE 0 END)
                AS missing_revenue,

            SUM(CASE WHEN budget IS NULL THEN 1 ELSE 0 END)
                AS missing_budget,

            SUM(CASE WHEN avg_rating IS NULL THEN 1 ELSE 0 END)
                AS missing_avg_rating,

            SUM(CASE WHEN rating_count IS NULL THEN 1 ELSE 0 END)
                AS missing_rating_count,

            SUM(CASE WHEN director IS NULL THEN 1 ELSE 0 END)
                AS missing_director,

            SUM(CASE WHEN keywords_list IS NULL THEN 1 ELSE 0 END)
                AS missing_keywords

        FROM movies_abt;
        """,
        connection,
    )

sql_validation

,abt_rows,unique_movie_ids,unique_tmdb_ids,missing_revenue,missing_budget,missing_avg_rating,missing_rating_count,missing_director,missing_keywords
0,9082,9082,9082,0,0,57,57,24,761


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [39]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.

data_dictionary.loc[
    data_dictionary["column_name"] == "movieId",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Unique MovieLens movie identifier",
    "movies_metadata",
    "Primary key used for movie-level analysis",
    "Used to aggregate ratings and identify movies"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "tmdbId",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "TMDB movie identifier",
    "movies_metadata",
    "Used to join credits and keywords tables",
    "Should be unique per movie"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "imdb_id",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "IMDb movie identifier",
    "movies_metadata",
    "Original source column",
    "Used for external movie references"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "title",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Movie title",
    "movies_metadata",
    "Original source column",
    ""
]

data_dictionary.loc[
    data_dictionary["column_name"] == "original_language",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Original language of the movie",
    "movies_metadata",
    "Original source column",
    ""
]

data_dictionary.loc[
    data_dictionary["column_name"] == "release_year",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Movie release year",
    "movies_metadata",
    "Extracted from release_date",
    ""
]

data_dictionary.loc[
    data_dictionary["column_name"] == "revenue",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Box office revenue reported for the movie",
    "movies_metadata",
    "Original source column",
    "Many observations contain a value of 0 which may represent missing or unreported financial information"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "avg_rating",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Average user rating for the movie",
    "ratings.csv",
    "Calculated as mean rating grouped by movieId",
    "Movies without ratings have missing values"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "rating_count",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Number of user ratings received",
    "ratings.csv",
    "Calculated as count of ratings grouped by movieId",
    "Some movies have very few ratings"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "rating_std",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Standard deviation of ratings",
    "ratings.csv",
    "Standard deviation grouped by movieId",
    "Measures variation in viewer opinions"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "director",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Primary director of the movie",
    "credits_curated.csv",
    "Joined on tmdbId",
    "Some values may be missing"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "keywords_list",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Keywords describing the movie",
    "keywords_curated.csv",
    "Joined on tmdbId",
    "Used to represent movie themes"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "release_date",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Movie release date",
    "movies_metadata",
    "Converted to datetime format",
    "Some missing values may exist"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "runtime",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Movie runtime in minutes",
    "movies_metadata",
    "Original source column",
    "Some values equal 0"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "budget",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Production budget of the movie",
    "movies_metadata",
    "Original source column",
    "Many records contain a value of 0"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "popularity",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "TMDB popularity score",
    "movies_metadata",
    "Original source column",
    ""
]

data_dictionary.loc[
    data_dictionary["column_name"] == "vote_average",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Average TMDB user score",
    "movies_metadata",
    "Original source column",
    ""
]

data_dictionary.loc[
    data_dictionary["column_name"] == "vote_count",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Number of TMDB user votes",
    "movies_metadata",
    "Original source column",
    ""
]

data_dictionary.loc[
    data_dictionary["column_name"] == "genres",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Movie genre information",
    "movies_metadata",
    "Original source column",
    "May contain multiple genres"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "production_companies",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Production companies associated with the movie",
    "movies_metadata",
    "Original source column",
    ""
]

data_dictionary.loc[
    data_dictionary["column_name"] == "production_countries",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Countries where the movie was produced",
    "movies_metadata",
    "Original source column",
    ""
]

data_dictionary.loc[
    data_dictionary["column_name"] == "adult",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Adult-content indicator",
    "movies_metadata",
    "Original source column",
    "Primarily used for filtering"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "status",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Movie release status",
    "movies_metadata",
    "Original source column",
    "Examples include Released and Post Production"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "genres_list",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "List of movie genres",
    "movies_metadata",
    "Derived from genres field",
    ""
]

data_dictionary.loc[
    data_dictionary["column_name"] == "primary_genre",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Primary genre assigned to the movie",
    "movies_metadata",
    "Derived from genres_list",
    "Used for simplified genre analysis"
]

data_dictionary.loc[
    data_dictionary["column_name"] == "top_cast",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = [
    "Primary cast members associated with the movie",
    "credits_curated.csv",
    "Joined on tmdbId",
    "May contain multiple actors"
]
display(data_dictionary)




,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,Unique MovieLens movie identifier,movies_metadata,Primary key used for movie-level analysis,Used to aggregate ratings and identify movies
1,tmdbId,int64,TMDB movie identifier,movies_metadata,Used to join credits and keywords tables,Should be unique per movie
2,imdb_id,str,IMDb movie identifier,movies_metadata,Original source column,Used for external movie references
3,title,str,Movie title,movies_metadata,Original source column,
4,original_title,str,,,,
5,original_language,str,Original language of the movie,movies_metadata,Original source column,
6,release_date,str,Movie release date,movies_metadata,Converted to datetime format,Some missing values may exist
7,runtime,float64,Movie runtime in minutes,movies_metadata,Original source column,Some values equal 0
8,budget,int64,Production budget of the movie,movies_metadata,Original source column,Many records contain a value of 0
9,revenue,float64,Box office revenue reported for the movie,movies_metadata,Original source column,Many observations contain a value of 0 which m...


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [8]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,6002,1994-01-01,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298936


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [9]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
